# 1. Importing libraries

In [1]:
import os
import re
import time

import shutil
import requests
import lxml
import json

import numpy as np
import pandas as pd

from bs4 import BeautifulSoup as bs
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service as ChromiumService
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
# Ensuring empty directory is created

if os.path.isdir('data'):
    shutil.rmtree('data')
os.makedirs('data/summaries')

# 2. Getting started with scrapping

In [3]:
# Parsing website skelton to bs4 object

BASE_URL = 'https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_'
chapter_urls = [BASE_URL + str(i) for i in range(0, 206)]
for url in chapter_urls[:5]:
    print(url)

https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_0
https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_1
https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_2
https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_3
https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_4


In [4]:
arcs = {
    "Final Selection Arc": (1,9),
    "Kidnapper's Bog Arc": (10,13),
    "Asakusa Arc": (14,19),
    "Tsuzumi Mansion Arc": (20,27),
    "Mount Natagumo Arc": (28,44),
    "Rehabilitation Training Arc": (45,53),
    "Mugen Train Arc": (54,66),
    "Entertainment District Arc": (67,97),
    "Swordsmith Village Arc": (98,127),
    "Hashira Training Arc": (128,139),
    "Infinity Castle Arc": (140,183),
    "Sunrise Countdown Arc": (184,205)
}

# 4. Defining helper functions

In [5]:
def get_summary(chapter_link, arc_name):
    r = requests.get(chapter_link)
    soup = bs(r.content, 'lxml')

    # All text paragraphs inside the main content
    content_div = soup.find('div', class_='mw-parser-output')
    if not content_div:
        print(f"No content found for {chapter_link}")
        return []

    paragraphs = content_div.find_all('p')
    text_list = []

    # Keywords to detect metadata block
    metadata_keywords = [
        "Chapter Information", "Volume", "Pages", "Date Released",
        "WSJ Issue", "Anime Episode(s)", "Kimetsu no Yaiba"
    ]

    for para in paragraphs:
        para_text = para.get_text().strip()
        if para_text == "":
            continue
        # Skip paragraphs that contain metadata keywords
        if any(keyword in para_text for keyword in metadata_keywords):
            continue

        # Replace non-breaking space \xa0 with normal space
        para_text = para_text.replace('\xa0', ' ')
        text_list.append(para_text)

    # Append text to file
    filename = f'data/summaries/{arc_name}.txt'
    with open(filename, 'a', encoding='utf-8') as f:
        f.write("\n".join(text_list) + "\n")

    return text_list  # list of story paragraphs

In [ ]:
linktest1 = "https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_2"
linktest2 = "https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_2"
arctest  = "Final Selection Arc"
get_summary(linktest1, arctest)
get_summary(linktest2, arctest)

In [6]:
def get_starring(chapter_link):
    r = requests.get(chapter_link)
    soup = bs(r.content, 'lxml')

    # Main content
    content_div = soup.select_one('#mw-content-text > div.mw-parser-output')
    if not content_div:
        return []

    # Try the scrollable div first
    char_div = content_div.find('div', style=lambda s: s and 'overflow:auto' in s)
    ul = None

    if char_div:
        ul = char_div.find('ul')
    else:
        # fallback: find all <ul> inside main content
        all_uls = content_div.find_all('ul')
        # Skip TOC / navigation by checking if multiple <ul> exist
        if len(all_uls) > 1:
            # pick the first <ul> with more than 1 capitalized name
            for candidate in all_uls:
                names = [li.get_text(strip=True) for li in candidate.find_all('li')]
                if any(re.match(r'[A-Z][a-z]+', n) for n in names):
                    ul = candidate
                    break
        elif all_uls:
            ul = all_uls[0]

    if not ul:
        return []  # nothing found

    characters = [li.get_text(strip=True) for li in ul.find_all('li')]
    return characters

chars = get_starring("https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_3")
print(chars)

['Sakonji Urokodaki', 'Giyu Tomioka', 'Tanjiro Kamado', 'Nezuko Kamado', 'Temple Demon']


# 5. Gathering each episodes summary and character names

In [7]:
episode_links, characters = {}, {}

for arc, (start, end) in arcs.items():
    
    episode_links[arc] = chapter_urls[start:end+1]
    
    characts = []
    
    for episode in episode_links[arc]:
        get_summary(episode, arc)      # Make sure your summary function is correct
        characts.append(get_starring(episode))
        
    characters[arc] = list(np.concatenate(characts))
    
    print(f"Arc '{arc}' :: Number of episodes recorded = {len(episode_links[arc])}")

# Optional: total episodes
total_eps = sum(len(eps) for eps in episode_links.values())
print(f"Total episodes recorded :: {total_eps}")

Arc 'Final Selection Arc' :: Number of episodes recorded = 9
Arc 'Kidnapper's Bog Arc' :: Number of episodes recorded = 4
Arc 'Asakusa Arc' :: Number of episodes recorded = 6
Arc 'Tsuzumi Mansion Arc' :: Number of episodes recorded = 8
Arc 'Mount Natagumo Arc' :: Number of episodes recorded = 17
Arc 'Rehabilitation Training Arc' :: Number of episodes recorded = 9
Arc 'Mugen Train Arc' :: Number of episodes recorded = 13
Arc 'Entertainment District Arc' :: Number of episodes recorded = 31
Arc 'Swordsmith Village Arc' :: Number of episodes recorded = 30
Arc 'Hashira Training Arc' :: Number of episodes recorded = 12
Arc 'Infinity Castle Arc' :: Number of episodes recorded = 44
Arc 'Sunrise Countdown Arc' :: Number of episodes recorded = 22
Total episodes recorded :: 205


In [ ]:
# chars = get_starring("https://kimetsu-no-yaiba.fandom.com/wiki/Chapter_3")
# print(chars)
for arc, (start, end) in arcs.items():
    print(arc, " ", len(characters[arc]))

In [8]:
with open('data/arcs_nd_chapter_links.txt', 'w') as file:
        file.write(str(episode_links))

# 6. Processing obtained character name info.

In [9]:
seas = []
chars = []

for arc in arcs.keys():
    seas += [arc] * len(characters[arc])
    chars += characters[arc]

character_df = pd.DataFrame({'Arc': seas, 'Character': chars})

In [10]:

character_df.to_csv('data/character_df.csv', index = False)
character_df.head()

,Arc,Character
0,Final Selection Arc,Nezuko Kamado
1,Final Selection Arc,Tanjiro Kamado
2,Final Selection Arc,Kie Kamado
3,Final Selection Arc,Shigeru Kamado
4,Final Selection Arc,Hanako Kamado


In [11]:
character_df.to_csv('data/character_df_cleaned.csv', index = False)
character_df.tail()

,Arc,Character
1851,Sunrise Countdown Arc,Kiriya Ubuyashiki(Photograph)
1852,Sunrise Countdown Arc,Kuina Ubuyashiki(Photograph)
1853,Sunrise Countdown Arc,Naho Takada(Photograph)
1854,Sunrise Countdown Arc,Sumi Nakahara(Photograph)
1855,Sunrise Countdown Arc,Kiyo Terauchi(Photograph)


In [12]:
print('Number of Characters recorded per arc'); print('-' * 40)
character_df.Arc.value_counts(sort = False)

Number of Characters recorded per arc
----------------------------------------


Arc
Final Selection Arc             63
Kidnapper's Bog Arc             32
Asakusa Arc                     48
Tsuzumi Mansion Arc             65
Mount Natagumo Arc             154
Rehabilitation Training Arc    124
Mugen Train Arc                107
Entertainment District Arc     229
Swordsmith Village Arc         235
Hashira Training Arc           144
Infinity Castle Arc            328
Sunrise Countdown Arc          327
Name: count, dtype: int64

In [13]:
# Cleaning character_df : Removing and stripping unwanted characters
CLeaning_character_df = character_df.copy()
CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : re.sub(u'\xa0', u' ', x).strip()) # https://stackoverflow.com/a/11566398
CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : re.sub(r'\t', ' ', x).strip())
CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : x.split(' as ')[-1].strip())
CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : re.sub(r'^as\s', ' ', x).strip())



In [14]:
print(CLeaning_character_df[CLeaning_character_df['Arc'] == 'Final Selection Arc'])

                    Arc          Character
0   Final Selection Arc      Nezuko Kamado
1   Final Selection Arc     Tanjiro Kamado
2   Final Selection Arc         Kie Kamado
3   Final Selection Arc     Shigeru Kamado
4   Final Selection Arc      Hanako Kamado
..                  ...                ...
58  Final Selection Arc  Sakonji Urokodaki
59  Final Selection Arc  Hotaru Haganezuka
60  Final Selection Arc  Matsuemon Tennoji
61  Final Selection Arc             Satoko
62  Final Selection Arc             Kazumi

[63 rows x 2 columns]


In [ ]:
for idx in [19, 20, 32, 1230]:
    print(character_df.iloc[idx][1])

In [15]:
# Processing character names

CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : re.sub(r'#\d*', ' ', x).strip()) # Removing '#' from name
CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : re.sub(r'\(.*\)', ' ', x).strip()) # Removing '()' and text inside those
CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : re.sub(r'CarWash Patron', 'Car Wash Patron', x).strip())
CLeaning_character_df.Character = CLeaning_character_df.Character.apply(lambda x : re.sub(r'\s+', ' ', x).strip()) # Removing unwanted extra white spaces

In [16]:
CLeaning_character_df.Character = (
    CLeaning_character_df.Character
    # Remove outer quotes if present
    .str.replace(r'^"(.*)"$', r'\1', regex=True)
    # Remove colon suffixes
    .str.replace(r':.*$', '', regex=True).str.strip()
    # Collapse "Reincarnation of ..." → "Reincarnation"
    .str.replace(r'^Reincarnation of.*', 'Reincarnation', regex=True)
    # Collapse "Descendant of ..." → "Descendant"
    .str.replace(r'^Descendant of.*', 'Descendant', regex=True)
)

In [17]:
CLeaning_character_df.Character = (
    CLeaning_character_df.Character
    # Remove outer quotes if present
    .str.replace(r'^"(.*)"$', r'\1', regex=True)
    # Replace specific Muzan variants
    .str.replace(r'^Muzan.*daughter.*', 'Muzan Kibutsuji', regex=True)
    .str.replace(r"^Muzan's\s*$", 'Muzan Kibutsuji', regex=True)
    # Remove colon suffixes
    .str.replace(r':.*$', '', regex=True).str.strip()
    # Collapse "Reincarnation of ..." → "Reincarnation"
    .str.replace(r'^Reincarnation of.*', 'Reincarnation', regex=True)
    # Collapse "Descendant of ..." → "Descendant"
    .str.replace(r'^Descendant of.*', 'Descendant', regex=True)
    # Replace unwanted noise with None
    .str.replace(r'^(Chapter Cover|Color Page|Twitter Icon|Bonus Manga.*|Chapter\s*\d+)$', 'None', regex=True)
)

# Manually fix the two entries
CLeaning_character_df.Character = CLeaning_character_df.Character.replace({
    'Kokushibo / Michikatsu Tsugikuni': 'Kokushibo',
    'Hakuji /Akaza': 'Akaza'
})

In [18]:
print(CLeaning_character_df.tail(20))

                        Arc          Character
1836  Sunrise Countdown Arc            Kotetsu
1837  Sunrise Countdown Arc              Makio
1838  Sunrise Countdown Arc          Hinatsuru
1839  Sunrise Countdown Arc   Shinjuro Rengoku
1840  Sunrise Countdown Arc           Takeuchi
1841  Sunrise Countdown Arc             Murata
1842  Sunrise Countdown Arc  Inosuke Hashibira
1843  Sunrise Countdown Arc     Tanjiro Kamado
1844  Sunrise Countdown Arc      Nezuko Kamado
1845  Sunrise Countdown Arc   Zenitsu Agatsuma
1846  Sunrise Countdown Arc      Kanao Tsuyuri
1847  Sunrise Countdown Arc        Aoi Kanzaki
1848  Sunrise Countdown Arc    Senjuro Rengoku
1849  Sunrise Countdown Arc       Giyu Tomioka
1850  Sunrise Countdown Arc  Kanata Ubuyashiki
1851  Sunrise Countdown Arc  Kiriya Ubuyashiki
1852  Sunrise Countdown Arc   Kuina Ubuyashiki
1853  Sunrise Countdown Arc        Naho Takada
1854  Sunrise Countdown Arc      Sumi Nakahara
1855  Sunrise Countdown Arc      Kiyo Terauchi


In [19]:
print(f'Total number of characters\t\t:: {len(CLeaning_character_df.Character)}')
print(f'Total number of unique characters\t:: {CLeaning_character_df.Character.nunique()}')

Total number of characters		:: 1856
Total number of unique characters	:: 179


In [20]:
CLeaning_character_df.head()

,Arc,Character
0,Final Selection Arc,Nezuko Kamado
1,Final Selection Arc,Tanjiro Kamado
2,Final Selection Arc,Kie Kamado
3,Final Selection Arc,Shigeru Kamado
4,Final Selection Arc,Hanako Kamado


In [21]:
CLeaning_character_df.to_csv('data/character_df_cleaned.csv', index = False)
CLeaning_character_df.tail()

,Arc,Character
1851,Sunrise Countdown Arc,Kiriya Ubuyashiki
1852,Sunrise Countdown Arc,Kuina Ubuyashiki
1853,Sunrise Countdown Arc,Naho Takada
1854,Sunrise Countdown Arc,Sumi Nakahara
1855,Sunrise Countdown Arc,Kiyo Terauchi


# 7. Next is What ?

- We have generated summaries all seasons using web scrapping in this part.
- In the next part file [Relationship Finder / Network Analysis](https://www.kaggle.com/code/jishnukoliyadan/relationship-finder-network-analysis), we will try to find the relationsship between characters using `spaCy` and `NetworkX` libraries.

# 8. Where to look for the code and IPYNB notebook?

<br><a href = 'https://github.com/jishnukoliyadan/the_breaking_bad_network' target = '_blank'><img src = 'https://raw.githubusercontent.com/jishnukoliyadan/usefull_items/master/svgs/GitHub_View_Source.svg' width = 25%></a><br>
<a href = 'https://nbviewer.org/github/jishnukoliyadan/the_breaking_bad_network/blob/master/Scrapper.ipynb' target = '_blank'><img src = 'https://raw.githubusercontent.com/jishnukoliyadan/usefull_items/master/svgs/NbViwer_View_In.svg' width = 25%></a><br>
<a href = 'https://colab.research.google.com/github/jishnukoliyadan/the_breaking_bad_network/blob/master/Scrapper.ipynb' target = '_blank'><img src = 'https://raw.githubusercontent.com/jishnukoliyadan/usefull_items/master/svgs/Colab_Run_In.svg' width = 25%></a><br>
<a href = 'https://www.kaggle.com/code/jishnukoliyadan/data-collection-scrapper' target = '_blank'><img src = 'https://raw.githubusercontent.com/jishnukoliyadan/usefull_items/master/svgs/Kaggle_View_On.svg' width = 25%></a>

# References

References
- Official documentations of all libraries
- [Fandom](https://www.fandom.com/)
- [Kaggle : data-collection-web-scrapping-tutorial](https://www.kaggle.com/code/jishnukoliyadan/data-collection-web-scrapping-tutorial)
- [DataCamp - A Network Analysis of Game of Thrones](https://www.datacamp.com/projects/76)

Query Resolution
- [GeeksforGeeks](https://www.geeksforgeeks.org/)
- [Stack Overflow](https://stackoverflow.com/)
    
Image Credits
- [IMDb.com](https://www.imdb.com/)